In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch as t
from transformer_lens import HookedTransformer
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from dadapy.data import Data
from collections import defaultdict


import transformer_lens.utils as utils
import einops

from joblib import Parallel, delayed
import pandas as pd

import plot_utils

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
# Saves computation time, since we don't need it for the contents of this notebook
t.set_grad_enabled(False)


In [3]:
# Load GPT-2 Small
model = HookedTransformer.from_pretrained("gpt2-small")

# Load prompt from Pile-10K (Prompt 3218)
pile_dataset = load_dataset("NeelNanda/pile-10k")

Loaded pretrained model gpt2-small into HookedTransformer


In [4]:
filtered_indices = np.load('filtered_indices.npy')


In [5]:
filtered_indices

array([   0,   19,   22, ..., 9989, 9991, 9998], dtype=int64)

In [6]:
pile_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'meta'],
        num_rows: 10000
    })
})

In [7]:
filtered_dataset = pile_dataset['train'][filtered_indices]

In [8]:
setname_to_indexlist = defaultdict(list)

In [9]:
for i, set_name in enumerate(filtered_dataset['meta']):
    setname_to_indexlist[set_name['pile_set_name']].append(i)

In [10]:
arxiv_indices, wiki_indices, math_indices = setname_to_indexlist['ArXiv'], setname_to_indexlist['Wikipedia (en)'], setname_to_indexlist['DM Mathematics']

In [11]:
arxiv_prompts, wiki_prompts, math_prompts = [filtered_dataset['text'][idx] for idx in arxiv_indices], [filtered_dataset['text'][idx] for idx in wiki_indices], [filtered_dataset['text'][idx] for idx in math_indices]


In [12]:
arxiv_prompts[:10]

["---\nabstract: 'The purpose of this article is to study the problem of finding sharp lower bounds for the norm of the product of polynomials in the ultraproducts of Banach spaces $(X_i)_{\\mathfrak U}$. We show that, under certain hypotheses, there is a strong relation between this problem and the same problem for the spaces $X_i$.'\naddress: 'IMAS-CONICET'\nauthor:\n- Jorge Tomás Rodríguez\ntitle: On the norm of products of polynomials on ultraproducts of Banach spaces\n---\n\nIntroduction\n============\n\nIn this article we study the factor problem in the context of ultraproducts of Banach spaces. This problem can be stated as follows: for a Banach space $X$ over a field ${\\mathbb K}$ (with ${\\mathbb K}={\\mathbb R}$ or ${\\mathbb K}={\\mathbb C}$) and natural numbers $k_1,\\cdots, k_n$ find the optimal constant $M$ such that, given any set of continuous scalar polynomials $P_1,\\cdots,P_n:X\\rightarrow {\\mathbb K}$, of degrees $k_1,\\cdots,k_n$; the inequality $$\\label{problem

In [13]:
def prompts_to_tokens(prompts):
    tokens = model.to_tokens(prompts, prepend_bos=True)
    print(tokens.shape)
    tokens = tokens[..., :512]
    return tokens

In [14]:
arxiv_tokens, wiki_tokens, math_tokens = prompts_to_tokens(arxiv_prompts), prompts_to_tokens(wiki_prompts), prompts_to_tokens(math_prompts)

torch.Size([87, 1024])
torch.Size([123, 1024])
torch.Size([99, 1024])


In [15]:
# Function to compute intrinsic dimension (ID) with dadapy
def compute_ids(full_reps):
    ids = []
    for rep in full_reps:
        _data = Data(coordinates=rep.cpu().detach().numpy(), maxk=100)
        ids.append(_data.return_id_scaling_gride(range_max=64))
    return np.array(ids)

In [16]:
def zero_attn_out_hook(attn_out, hook):
    # print(attn_out.shape)
    return t.zeros_like(attn_out)

zero_attn_hooks = [
            # Changed hook point to blocks.{layer}.attn.hook_result
            (utils.get_act_name("attn_out", layer), zero_attn_out_hook)
            for layer in range(model.cfg.n_layers)
        ]

In [17]:
def summary_statistics(idim, n_bootstrap=1000):
    y_data = np.array(idim)
    
    num_lines, num_points = y_data.shape
    # --- 2. Calculate Average and Bootstrap Standard Error ---

    # Calculate the average y-value for each x-point across all 10 lines.
    y_average = np.mean(y_data, axis=0)

    # Calculate error bars using bootstrapping
    bootstrap_std_err = np.zeros(num_points)

    for j in range(num_points): # For each x-point (column)
        point_data = y_data[:, j] # Get the y-values from all lines for this x-point
        bootstrap_means = np.zeros(n_bootstrap)

        for i in range(n_bootstrap):
            # Resample the data *with replacement*
            resampled_data = np.random.choice(point_data, size=num_lines, replace=True)
            # Calculate the mean of the resampled data
            bootstrap_means[i] = np.mean(resampled_data)

        # Calculate the standard deviation of the bootstrap means
        # This serves as the bootstrapped estimate of the standard error.
        # Alternatively, one could calculate a confidence interval (e.g., 2.5th and 97.5th percentiles)
        # from bootstrap_means for potentially asymmetric error bars.
        bootstrap_std_err[j] = np.std(bootstrap_means, ddof=1)
    return y_average, bootstrap_std_err
        

In [30]:
idx_range = np.arange(0, 512+1, 128)

In [19]:
def residuals_to_idim_prompt_and_free(accumulated_residual, accumulated_residual_free, N=10):
    
    print(accumulated_residual.shape)
    idim_accumulated = []
    idim_accumulated_free = []
    
    
    for idx in range(len(arxiv_tokens[:N])):
        print(accumulated_residual[:, idx].shape)
        ids = compute_ids(accumulated_residual[:, idx])
        print(ids.shape)
        ids_free = compute_ids(accumulated_residual_free[:, idx])
        idim_accumulated.append(ids[:, 0, 1])
        idim_accumulated_free.append(ids_free[:, 0, 1])
    # print(accumulated_residual.shape)
    return idim_accumulated, idim_accumulated_free
        

In [20]:
def get_interaction_values(idim, idim_free):
    idim_interaction = np.array(idim_free) - np.array(idim)
    g_interaction = idim_interaction / np.array(idim)
    return idim_interaction, g_interaction

In [ ]:
idim_avg_list = []
idim_err_list = []
idim_free_avg_list = []
idim_free_err_list = []
idim_interaction_avg_list = []
idim_interaction_err_list = []
idim_g_avg_list = []
idim_g_err_list = []
def tokens_to_idx_range_graph_values(tokens, N=10, idx_range = np.arange(0, 512+1, 128)):
    logits, cache = model.run_with_cache(tokens[:N])
    accumulated_residual, labels = cache.accumulated_resid(layer = -1, incl_mid=True,\
                                                return_labels = True)
    
    print(accumulated_residual.shape)
    with model.hooks(fwd_hooks=zero_attn_hooks):
        logits_free, cache_free = model.run_with_cache(tokens[:N])
        accumulated_residual_free, _ = cache_free.accumulated_resid(layer = -1, incl_mid=True,\
                                                return_labels = True)
    
    for start_idx, end_idx in zip(idx_range[:-1], idx_range[1:]):
        print(start_idx, end_idx)
        idim, idim_free = residuals_to_idim_prompt_and_free(accumulated_residual[:,:,start_idx:end_idx, :], accumulated_residual_free[:,:,start_idx:end_idx, :])
        avg, err = summary_statistics(idim)
        
        idim_interaction, g_interaction = get_interaction_values(idim, idim_free)
        

        idim_free_avg, idim_free_err = summary_statistics(idim_free)
        idim_interaction_avg, idim_interaction_err = summary_statistics(idim_interaction)
        idim_g_avg, idim_g_err = summary_statistics(g_interaction)
        
        idim_avg_list.append(avg)
        idim_err_list.append(err)
        idim_free_avg_list.append(idim_free_avg)
        idim_free_err_list.append(idim_free_err)
        idim_interaction_avg_list.append(idim_interaction_avg)
        idim_interaction_err_list.append(idim_interaction_err)
        idim_g_avg_list.append(idim_g_avg)
        idim_g_err_list.append(idim_g_err)
    return labels

        
        

In [ ]:
labels = tokens_to_idx_range_graph_values(arxiv_tokens)

torch.Size([25, 10, 512, 768])
0 128
torch.Size([25, 10, 128, 768])
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
128 256
torch.Size([25, 10, 128, 768])
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
256 384
torch.Size([25, 10, 128, 768])
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 12

In [23]:
idim_free_avg_list

[array([6.12, 6.12, 3.34, 3.34, 3.31, 3.31, 3.97, 3.97, 4.37, 4.37, 4.37,
        4.37, 4.4 , 4.4 , 4.46, 4.46, 4.44, 4.44, 4.37, 4.37, 4.18, 4.18,
        3.96, 3.96, 3.7 ]),
 array([5.38, 5.38, 3.56, 3.56, 3.45, 3.45, 3.52, 3.52, 3.51, 3.51, 3.45,
        3.45, 3.45, 3.45, 3.41, 3.41, 3.35, 3.35, 3.26, 3.26, 3.13, 3.13,
        3.02, 3.02, 2.81]),
 array([3.57, 3.57, 2.37, 2.37, 2.34, 2.34, 2.43, 2.43, 2.43, 2.43, 2.41,
        2.41, 2.41, 2.41, 2.39, 2.39, 2.36, 2.36, 2.3 , 2.3 , 2.22, 2.22,
        2.15, 2.15, 2.06]),
 array([3.09, 3.09, 1.95, 1.95, 1.92, 1.92, 1.97, 1.97, 1.97, 1.97, 1.96,
        1.96, 1.97, 1.97, 1.96, 1.96, 1.95, 1.95, 1.91, 1.91, 1.86, 1.86,
        1.82, 1.82, 1.77])]

In [28]:
import plotly.graph_objects as go
import plotly
from plotly.colors import qualitative
colors = qualitative.Plotly # Get the default sequence
D3colors = qualitative.D3

In [29]:
def get_alpha_color_string(hex_color, alpha = 0.5):
    rgb_tuple = plotly.colors.hex_to_rgb(hex_color)
    return f'rgba({rgb_tuple[0]}, {rgb_tuple[1]}, {rgb_tuple[2]}, {alpha})'

In [31]:
range_list = [[start_idx, end_idx] for start_idx, end_idx in zip(idx_range[:-1], idx_range[1:])]

In [36]:
# Initialize a Plotly Figure object
fig = go.Figure()



# Example: Get the first color


# Add the main trace: the average line with error bars
for i in range(4):
    fig.add_trace(go.Scatter(
        x=labels,
        y=idim_avg_list[i],
        mode='lines+markers',  # Display both the line connecting points and markers at each point
        name=f'Arxiv Prompt token idx range ({range_list[i][0]},{range_list[i][1]})',   # Name for the legend
        line=dict(color=colors[i], width=2), # Style the average line
        marker=dict(size=5, color=colors[i]), # Style the markers
        error_y=dict(
            type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
            array=idim_err_list[i],   # Provide the calculated standard error values for the error bars
            visible=True,      # Make the error bars visible
            thickness=1,       # Thickness of the error bar lines
            width=3,           # Width of the e rror bar caps
            color=get_alpha_color_string(colors[i]) # Color of error bars (royalblue with some transparency)
        )
    ))

    fig.add_trace(go.Scatter(
        x=labels,
        y=idim_free_avg_list[i],
        mode='lines+markers',  # Display both the line connecting points and markers at each point
        name=f'Arxiv Free token idx range ({range_list[i][0]},{range_list[i][1]})',   # Name for the legend
        line=dict(color=D3colors[i], width=2), # Style the average line
        marker=dict(size=5, color=D3colors[i]), # Style the markers
        error_y=dict(
            type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
            array=idim_free_err_list[i],   # Provide the calculated standard error values for the error bars
            visible=True,      # Make the error bars visible
            thickness=1,       # Thickness of the error bar lines
            width=3,           # Width of the error bar caps
            color=get_alpha_color_string(D3colors[i]) # Color of error bars (royalblue with some transparency)
        )
    ))



# --- Optional: Add the original individual lines for context (commented out by default) ---
# Uncomment the following loop if you want to see the underlying data lines.
# for i in range(num_lines):
#     fig.add_trace(go.Scatter(
#         x=x_values,
#         y=y_data[i, :],
#         mode='lines',
#         name=f'Line {i+1}',
#         line=dict(width=0.5, color='rgba(128, 128, 128, 0.4)'), # Thin, semi-transparent grey lines
#         showlegend=False # Hide these individual lines from the main legend
#     ))

# --- 4. Customize the Plot Layout ---
fig.update_layout(font=dict(size=18)) 
fig.update_layout(
    title='Average of 10 Prompts', # Plot title
    xaxis_title='Layers',                 # X-axis label
    yaxis_title='IDim',                               # Y-axis label
    legend_title='Prompt and IDim Type',                               # Title for the legend box
    hovermode='x unified',                               # Show hover info for all traces at a given x-value
    template='plotly_white',
    showlegend=True,
)

#fig.write_image('IDIM_prompt_categories.png')
# --- 5. Show the Plot ---

# Display the figure. This will typically open it in a browser window
# or display it in the output cell of a Jupyter notebook.
fig.show()


In [37]:
# Initialize a Plotly Figure object
fig = go.Figure()



# Example: Get the first color


# Add the main trace: the average line with error bars
for i in range(4):
    fig.add_trace(go.Scatter(
        x=labels,
        y=idim_interaction_avg_list[i],
        mode='lines+markers',  # Display both the line connecting points and markers at each point
        name=f'Arxiv interaction token idx range ({range_list[i][0]},{range_list[i][1]})',   # Name for the legend
        line=dict(color=colors[i], width=2), # Style the average line
        marker=dict(size=5, color=colors[i]), # Style the markers
        error_y=dict(
            type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
            array=idim_interaction_err_list[i],   # Provide the calculated standard error values for the error bars
            visible=True,      # Make the error bars visible
            thickness=1,       # Thickness of the error bar lines
            width=3,           # Width of the e rror bar caps
            color=get_alpha_color_string(colors[i]) # Color of error bars (royalblue with some transparency)
        )
    ))





# --- Optional: Add the original individual lines for context (commented out by default) ---
# Uncomment the following loop if you want to see the underlying data lines.
# for i in range(num_lines):
#     fig.add_trace(go.Scatter(
#         x=x_values,
#         y=y_data[i, :],
#         mode='lines',
#         name=f'Line {i+1}',
#         line=dict(width=0.5, color='rgba(128, 128, 128, 0.4)'), # Thin, semi-transparent grey lines
#         showlegend=False # Hide these individual lines from the main legend
#     ))

# --- 4. Customize the Plot Layout ---
fig.update_layout(font=dict(size=18)) 
fig.update_layout(
    title='Average of 10 Prompts', # Plot title
    xaxis_title='Layers',                 # X-axis label
    yaxis_title='IDim',                               # Y-axis label
    legend_title='Prompt and IDim Type',                               # Title for the legend box
    hovermode='x unified',                               # Show hover info for all traces at a given x-value
    template='plotly_white',
    showlegend=True,
)

#fig.write_image('IDIM_prompt_categories.png')
# --- 5. Show the Plot ---

# Display the figure. This will typically open it in a browser window
# or display it in the output cell of a Jupyter notebook.
fig.show()


In [38]:
labels = tokens_to_idx_range_graph_values(math_tokens)

torch.Size([25, 10, 512, 768])
0 128
torch.Size([25, 10, 128, 768])
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
128 256
torch.Size([25, 10, 128, 768])
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
256 384
torch.Size([25, 10, 128, 768])
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 128, 768])
(25, 3, 6)
torch.Size([25, 12

In [39]:
# Initialize a Plotly Figure object
fig = go.Figure()



# Example: Get the first color


# Add the main trace: the average line with error bars
for i in range(4):
    fig.add_trace(go.Scatter(
        x=labels,
        y=idim_avg_list[i],
        mode='lines+markers',  # Display both the line connecting points and markers at each point
        name=f'Math Prompt token idx range ({range_list[i][0]},{range_list[i][1]})',   # Name for the legend
        line=dict(color=colors[i], width=2), # Style the average line
        marker=dict(size=5, color=colors[i]), # Style the markers
        error_y=dict(
            type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
            array=idim_err_list[i],   # Provide the calculated standard error values for the error bars
            visible=True,      # Make the error bars visible
            thickness=1,       # Thickness of the error bar lines
            width=3,           # Width of the e rror bar caps
            color=get_alpha_color_string(colors[i]) # Color of error bars (royalblue with some transparency)
        )
    ))

    fig.add_trace(go.Scatter(
        x=labels,
        y=idim_free_avg_list[i],
        mode='lines+markers',  # Display both the line connecting points and markers at each point
        name=f'Math Free token idx range ({range_list[i][0]},{range_list[i][1]})',   # Name for the legend
        line=dict(color=D3colors[i], width=2), # Style the average line
        marker=dict(size=5, color=D3colors[i]), # Style the markers
        error_y=dict(
            type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
            array=idim_free_err_list[i],   # Provide the calculated standard error values for the error bars
            visible=True,      # Make the error bars visible
            thickness=1,       # Thickness of the error bar lines
            width=3,           # Width of the error bar caps
            color=get_alpha_color_string(D3colors[i]) # Color of error bars (royalblue with some transparency)
        )
    ))



# --- Optional: Add the original individual lines for context (commented out by default) ---
# Uncomment the following loop if you want to see the underlying data lines.
# for i in range(num_lines):
#     fig.add_trace(go.Scatter(
#         x=x_values,
#         y=y_data[i, :],
#         mode='lines',
#         name=f'Line {i+1}',
#         line=dict(width=0.5, color='rgba(128, 128, 128, 0.4)'), # Thin, semi-transparent grey lines
#         showlegend=False # Hide these individual lines from the main legend
#     ))

# --- 4. Customize the Plot Layout ---
fig.update_layout(font=dict(size=18)) 
fig.update_layout(
    title='Average of 10 Prompts', # Plot title
    xaxis_title='Layers',                 # X-axis label
    yaxis_title='IDim',                               # Y-axis label
    legend_title='Prompt and IDim Type',                               # Title for the legend box
    hovermode='x unified',                               # Show hover info for all traces at a given x-value
    template='plotly_white',
    showlegend=True,
)

#fig.write_image('IDIM_prompt_categories.png')
# --- 5. Show the Plot ---

# Display the figure. This will typically open it in a browser window
# or display it in the output cell of a Jupyter notebook.
fig.show()


In [40]:
# Initialize a Plotly Figure object
fig = go.Figure()



# Example: Get the first color


# Add the main trace: the average line with error bars
for i in range(4):
    fig.add_trace(go.Scatter(
        x=labels,
        y=idim_interaction_avg_list[i],
        mode='lines+markers',  # Display both the line connecting points and markers at each point
        name=f'Math interaction token idx range ({range_list[i][0]},{range_list[i][1]})',   # Name for the legend
        line=dict(color=colors[i], width=2), # Style the average line
        marker=dict(size=5, color=colors[i]), # Style the markers
        error_y=dict(
            type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
            array=idim_interaction_err_list[i],   # Provide the calculated standard error values for the error bars
            visible=True,      # Make the error bars visible
            thickness=1,       # Thickness of the error bar lines
            width=3,           # Width of the e rror bar caps
            color=get_alpha_color_string(colors[i]) # Color of error bars (royalblue with some transparency)
        )
    ))





# --- Optional: Add the original individual lines for context (commented out by default) ---
# Uncomment the following loop if you want to see the underlying data lines.
# for i in range(num_lines):
#     fig.add_trace(go.Scatter(
#         x=x_values,
#         y=y_data[i, :],
#         mode='lines',
#         name=f'Line {i+1}',
#         line=dict(width=0.5, color='rgba(128, 128, 128, 0.4)'), # Thin, semi-transparent grey lines
#         showlegend=False # Hide these individual lines from the main legend
#     ))

# --- 4. Customize the Plot Layout ---
fig.update_layout(font=dict(size=18)) 
fig.update_layout(
    title='Average of 10 Prompts', # Plot title
    xaxis_title='Layers',                 # X-axis label
    yaxis_title='IDim',                               # Y-axis label
    legend_title='Prompt and IDim Type',                               # Title for the legend box
    hovermode='x unified',                               # Show hover info for all traces at a given x-value
    template='plotly_white',
    showlegend=True,
)

#fig.write_image('IDIM_prompt_categories.png')
# --- 5. Show the Plot ---

# Display the figure. This will typically open it in a browser window
# or display it in the output cell of a Jupyter notebook.
fig.show()
